# Model Building v2 (Dual-Method, Anti-Leakage)

本 notebook 依照 model bulilding.md 實作：
1. 方法 A（可解釋特徵）
2. 方法 B（語義特徵 + 可解釋特徵）
3. 所有特徵選擇放入 Pipeline（每 fold 獨立）
4. 使用 Stratified 10-Fold 比較，最後選最佳模型產出 submission

In [62]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('HAS_XGB:', HAS_XGB)
print('HAS_LGB:', HAS_LGB)

HAS_XGB: True
HAS_LGB: True


## Data Loading

In [63]:
DATA_DIR = Path('./data')
TRAIN_PATH = DATA_DIR / 'boy or girl 2025 train_missingValue.csv'
TEST_PATH = DATA_DIR / 'boy or girl 2025 test no ans_missingValue.csv'

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(f'Missing data files:\n{TRAIN_PATH}\n{TEST_PATH}')

df_train_full = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

df_train, df_holdout = train_test_split(
    df_train_full,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_train_full['gender']
 )

print('Train full shape:', df_train_full.shape)
print('Train 80% shape:', df_train.shape)
print('Holdout 20% shape:', df_holdout.shape)
print('Test shape:', df_test.shape)
print('Train full gender distribution:', df_train_full['gender'].value_counts().to_dict())
print('Train 80% gender distribution:', df_train['gender'].value_counts().to_dict())
print('Holdout 20% gender distribution:', df_holdout['gender'].value_counts().to_dict())

Train full shape: (423, 11)
Train 80% shape: (338, 11)
Holdout 20% shape: (85, 11)
Test shape: (426, 11)
Train full gender distribution: {1: 316, 2: 107}
Train 80% gender distribution: {1: 253, 2: 85}
Holdout 20% gender distribution: {1: 63, 2: 22}


In [ ]:
STAR_ELEMENT_MAP = {
    'aries': 'fire', 'leo': 'fire', 'sagittarius': 'fire',
    'taurus': 'earth', 'virgo': 'earth', 'capricorn': 'earth',
    'gemini': 'air', 'libra': 'air', 'aquarius': 'air',
    'cancer': 'water', 'scorpio': 'water', 'pisces': 'water',
    '牡羊座': 'fire', '獅子座': 'fire', '射手座': 'fire',
    '金牛座': 'earth', '處女座': 'earth', '摩羯座': 'earth',
    '雙子座': 'air', '天秤座': 'air', '水瓶座': 'air',
    '巨蟹座': 'water', '天蠍座': 'water', '雙魚座': 'water'
}

MALE_WORDS = ['handsome', 'basketball', 'loser', 'game', 'esports']
FEMALE_WORDS = ['beautiful', 'cute', 'makeup', 'skincare', 'perfume']

class FeatureBuilder(BaseEstimator, TransformerMixin):

    def __init__(self, method='A', n_semantic=20):
        self.method = method
        self.n_semantic = n_semantic
        self.vectorizer_ = None
        self.svd_ = None
        self.height_bin_edges_ = None
        self.height_median_ = 165.0
        self.weight_median_ = 60.0
        self.fb_friends_median_ = 100.0
        self.yt_median_ = 0.0

    def _to_df(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X)

    def _clean_outliers(self, df):
        out = df.copy()
        if 'height' in out.columns:
            out.loc[(out['height'] < 140) | (out['height'] > 200), 'height'] = np.nan
        if 'weight' in out.columns:
            out.loc[(out['weight'] < 35) | (out['weight'] > 120), 'weight'] = np.nan
        return out

    def fit(self, X, y=None):
        X = self._to_df(X)
        X_cleaned = self._clean_outliers(X)
        
        # Compute height bin edges
        height = pd.to_numeric(X_cleaned.get('height', pd.Series(dtype=float)), errors='coerce')
        self.height_median_ = height.median() if not height.isna().all() else 165.0
        
        h1, h2 = height.quantile(0.33), height.quantile(0.67)
        if pd.isna(h1) or pd.isna(h2) or h1 >= h2:
            h1, h2 = 160.0, 175.0
        self.height_bin_edges_ = (-np.inf, h1, h2, np.inf)

        # Store weight median
        weight = pd.to_numeric(X_cleaned.get('weight', pd.Series(dtype=float)), errors='coerce')
        self.weight_median_ = weight.median() if not weight.isna().all() else 60.0

        # Store fb_friends and yt medians (for missing value imputation)
        fb = pd.to_numeric(X_cleaned.get('fb_friends', pd.Series(dtype=float)), errors='coerce')
        fb = fb[(fb >= 0) & (fb <= 5000)]  # Compute median only within valid bounds
        self.fb_friends_median_ = fb.median() if not fb.isna().all() else 100.0

        yt = pd.to_numeric(X_cleaned.get('yt', pd.Series(dtype=float)), errors='coerce')
        yt = yt[(yt >= 0) & (yt <= 1000000)]  # Compute median only within valid bounds
        self.yt_median_ = yt.median() if not yt.isna().all() else 0.0

        # Semantic features for Method B
        if self.method == 'B':
            corpus = X.get('self_intro', pd.Series('', index=X.index)).fillna('').astype(str)
            self.vectorizer_ = TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=2)
            tfidf = self.vectorizer_.fit_transform(corpus)
            comp = min(self.n_semantic, max(2, tfidf.shape[1] - 1)) if tfidf.shape[1] > 2 else 2
            self.svd_ = TruncatedSVD(n_components=comp, random_state=RANDOM_SEED)
            self.svd_.fit(tfidf)

        return self

    def transform(self, X):
        X = self._to_df(X)
        out = self._clean_outliers(X)

        # Remove excluded features
        drop_cols = [c for c in ['iq', 'phone_os', 'sleepiness','yt'] if c in out.columns]
        if drop_cols:
            out = out.drop(columns=drop_cols)

        # Basic bounds (fixed reasonable limits, not statistical)
        basic_bounds = {
            'height': (140, 200),      # cm
            'weight': (35, 120),       # kg
            'fb_friends': (0, 5000),   # 0-5000 friends
            
        }

        # Convert to numeric, apply bounds, and handle missing values
        for c in ['height', 'weight', 'fb_friends']:
            if c in out.columns:
                out[c] = pd.to_numeric(out[c], errors='coerce')
                # Remove non-finite values (NaN, inf, -inf)
                out.loc[~np.isfinite(out[c]), c] = np.nan
                # Apply basic bounds: set out-of-range values to NaN
                if c in basic_bounds:
                    lower, upper = basic_bounds[c]
                    out.loc[(out[c] < lower) | (out[c] > upper), c] = np.nan

        # Compute BMI
        temp_h = out['height'].fillna(self.height_median_) / 100.0
        temp_w = out['weight'].fillna(self.weight_median_)
        out['bmi'] = temp_w / (temp_h ** 2)
        
        out['is_tall'] = (out['height'] >= 170).astype(int)
        out['height_pct'] = out['height'].rank(pct=True)

        edges = self.height_bin_edges_ if self.height_bin_edges_ is not None else (-np.inf, 160, 175, np.inf)
        out['height_bin'] = pd.cut(out['height'], bins=edges, labels=['low', 'mid', 'high'])

        # Zodiac features
        star = out.get('star_sign', pd.Series('', index=out.index)).fillna('Unknown').astype(str).str.strip().str.lower()
        out['star_element'] = star.map(STAR_ELEMENT_MAP).fillna('unknown')

        # Textual features
        intro = out.get('self_intro', pd.Series('', index=out.index)).fillna('').astype(str)
        out['has_intro'] = (intro.str.strip() != '').astype(int)
        out['intro_len'] = intro.str.len().astype(float)
        out['uses_emoji'] = intro.str.contains(r'[\U0001F300-\U0001FAFF❤️]').astype(int)
        out['tilde_count'] = intro.str.count(r'[~～]').astype(float)
        out['exclamation_count'] = intro.str.count(r'!').astype(float)

        intro_lc = intro.str.lower()
        male_hits = intro_lc.apply(lambda s: sum(int(w in s) for w in MALE_WORDS))
        female_hits = intro_lc.apply(lambda s: sum(int(w in s) for w in FEMALE_WORDS))
        out['keyword_score'] = (male_hits - female_hits).astype(float)

        # Semantic features
        if self.method == 'B' and self.vectorizer_ is not None and self.svd_ is not None:
            tfidf = self.vectorizer_.transform(intro)
            sem = self.svd_.transform(tfidf)
            for i in range(sem.shape[1]):
                out[f'sem_{i}'] = sem[:, i]



        return out

In [139]:
def build_preprocessor(feature_cols):
    numeric_cols = [
        'height', 'weight', 'fb_friends', 'bmi', 'is_tall',
        'height_pct', 'has_intro', 'intro_len', 'uses_emoji', 'tilde_count',
        'exclamation_count', 'keyword_score'
    ]
    numeric_cols += [c for c in feature_cols if c.startswith('sem_')]

    # Excluded from modeling: iq, phone_os, sleepiness, yt.
    categorical_cols = ['star_element', 'height_bin']

    numeric_cols = [c for c in numeric_cols if c in feature_cols]
    categorical_cols = [c for c in categorical_cols if c in feature_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_cols)
        ],
        remainder='drop'
    )
    return preprocessor


def make_method_pipeline(method='A', model_name='xgb', selector_name='l1', scale_pos_weight=1.0):
    feat = FeatureBuilder(method=method, n_semantic=20)

    # Build a temporary transformed frame to determine available columns.
    temp = feat.fit_transform(df_train.drop(columns=['gender']))
    feature_cols = temp.columns.tolist()
    preprocessor = build_preprocessor(feature_cols)

    selector_l1 = SelectFromModel(
        LogisticRegression(
            penalty='l1', solver='liblinear', class_weight='balanced',
            C=0.5, random_state=RANDOM_SEED, max_iter=3000
        )
    )

    selector_rfe = RFE(
        estimator=LogisticRegression(
            penalty='l2', solver='liblinear', class_weight='balanced',
            random_state=RANDOM_SEED, max_iter=3000
        ),
        n_features_to_select=20,
        step=0.2
    )

    if model_name == 'xgb':
        if not HAS_XGB:
            raise RuntimeError('xgboost is not installed')
        clf = xgb.XGBClassifier(
            n_estimators=320,
            learning_rate=0.04,
            max_depth=5,
            min_child_weight=6,
            reg_alpha=1.5,
            reg_lambda=5.0,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='binary:logistic',
            eval_metric='auc',
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED
        )
    elif model_name == 'lgb':
        if not HAS_LGB:
            raise RuntimeError('lightgbm is not installed')
        clf = lgb.LGBMClassifier(
            n_estimators=400,
            learning_rate=0.04,
            max_depth=5,
            num_leaves=31,
            min_child_samples=20,
            reg_alpha=1.0,
            reg_lambda=5.0,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            verbose=-1
        )
    elif model_name == 'rf':
        clf = RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=4,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')

    if selector_name == 'l1':
        selector = selector_l1
    elif selector_name == 'rfe':
        selector = selector_rfe
    else:
        raise ValueError(f'Unsupported selector_name: {selector_name}')

    pipe = Pipeline([
        ('feature', feat),
        ('preprocess', preprocessor),
        ('selector', selector),
        ('model', clf)
    ])
    return pipe

In [ ]:
# ========== 10-Fold CV candidate model comparison ==========
print("=" * 80)
print("10-Fold Stratified CV: 候選模型比對 (Method A vs B)")
print("=" * 80)

# Compute class balance weight
n_neg = (y_bin == 0).sum()
n_pos = (y_bin == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f"\nscale_pos_weight (from train 80%): {scale_pos_weight:.4f}")

# Define scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'f1_macro': 'f1_macro',
    'accuracy': 'accuracy'
}

# CV configuration
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

# Define all 12 candidate models
candidates = []

for method in ['A', 'B']:
    for model_name in ['xgb', 'lgb', 'rf']:
        for selector_name in ['l1', 'rfe']:
            candidates.append({
                'method': method,
                'model': model_name,
                'selector': selector_name
            })

results = []

for config in candidates:
    method = config['method']
    model_name = config['model']
    selector_name = config['selector']
    
    pipe = make_method_pipeline(
        method=method,
        model_name=model_name,
        selector_name=selector_name,
        scale_pos_weight=scale_pos_weight
    )
    
    cv_result = cross_validate(pipe, X_raw, y_bin, cv=cv, scoring=scoring)
    
    results.append({
        'name': f"{method}_{model_name}_{selector_name}",
        'method': method,
        'roc_auc': cv_result['test_roc_auc'],
        'f1_macro': cv_result['test_f1_macro'],
        'accuracy': cv_result['test_accuracy']
    })

# Aggregate statistics
results_df = pd.DataFrame([
    {
        'name': r['name'],
        'method': r['method'],
        'roc_auc_mean': r['roc_auc'].mean(),
        'roc_auc_std': r['roc_auc'].std(),
        'f1_macro_mean': r['f1_macro'].mean(),
        'f1_macro_std': r['f1_macro'].std(),
        'accuracy_mean': r['accuracy'].mean(),
        'accuracy_std': r['accuracy'].std()
    }
    for r in results
])

# Rank by F1-macro
results_df['f1_rank'] = results_df['f1_macro_mean'].rank(ascending=False)
results_df = results_df.sort_values('f1_rank')

print("\n=== Candidate Ranking (CV on train 80%) ===\n")
print(results_df[['name', 'method', 'f1_macro_mean', 'f1_macro_std', 'accuracy_mean']].to_string(index=False))

# Select best candidate: prefer Method B (semantic features) when close
best_row = results_df.iloc[0]
best_name = best_row['name']

# Check whether B models are close enough to the best
b_candidates = results_df[results_df['method'] == 'B'].head(3)
if len(b_candidates) > 0:
    best_b = b_candidates.iloc[0]
    # If B lower bound (mean-std) is within competitive range, prefer B
    if best_b['f1_macro_mean'] - best_b['f1_macro_std'] >= best_row['f1_macro_mean'] - best_row['f1_macro_std'] - 0.01:
        best_name = best_b['name']
        print(f"\n→ Semantic features (Method B) still competitive, choosing {best_name}")
    else:
        print(f"\n→ Method A outperforms semantic features, choosing {best_name}")
else:
    print(f"\n→ Selecting best candidate: {best_name}")

cv_a = {r['name']: r for r in results}
cv_b = results_df.set_index('name').to_dict('index')

10-Fold Stratified CV: 候選模型比對 (Method A vs B)

scale_pos_weight (from train 80%): 2.9765

=== Candidate Ranking (CV on train 80%) ===

     name method  f1_macro_mean  f1_macro_std  accuracy_mean
 A_lgb_l1      A       0.842201      0.053730       0.878788
 B_lgb_l1      B       0.842201      0.053730       0.878788
A_lgb_rfe      A       0.838854      0.052144       0.875847
A_xgb_rfe      A       0.834720      0.053876       0.867023
 A_rf_rfe      A       0.822601      0.059233       0.860963
 A_xgb_l1      A       0.818881      0.063611       0.855080
 B_xgb_l1      B       0.818881      0.063611       0.855080
  A_rf_l1      A       0.817843      0.066246       0.858111
  B_rf_l1      B       0.817843      0.066246       0.858111
B_xgb_rfe      B       0.787583      0.075135       0.825312
B_lgb_rfe      B       0.784830      0.081211       0.828342
 B_rf_rfe      B       0.780016      0.068736       0.828431

→ Semantic features (Method B) still competitive, choosing B_lgb_l1


In [ ]:
# Export experiment results table
if 'results_df' not in globals():
    raise RuntimeError('results_df 不存在，請先執行前面的 CV 實驗程式格。')

export_cols = [
    'name', 'method',
    'f1_macro_mean', 'f1_macro_std',
    'accuracy_mean', 'accuracy_std',
    'roc_auc_mean', 'roc_auc_std'
]

experiment_table = (
    results_df[export_cols]
    .sort_values(['f1_macro_mean', 'accuracy_mean'], ascending=[False, False])
    .reset_index(drop=True)
    .copy()
)

# Round numeric columns for easier reading/reporting
for c in ['f1_macro_mean', 'f1_macro_std', 'accuracy_mean', 'accuracy_std', 'roc_auc_mean', 'roc_auc_std']:
    experiment_table[c] = experiment_table[c].round(6)

out_csv = 'experiment_results_table.csv'
experiment_table.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f'Saved: {out_csv}')
print('\nTop 10 experiment results:')
print(experiment_table.head(10).to_string(index=False))

Saved: experiment_results_table.csv

Top 10 experiment results:
     name method  f1_macro_mean  f1_macro_std  accuracy_mean  accuracy_std  roc_auc_mean  roc_auc_std
 A_lgb_l1      A       0.842201      0.053730       0.878788      0.042561      0.896991     0.060768
 B_lgb_l1      B       0.842201      0.053730       0.878788      0.042561      0.896991     0.060768
A_lgb_rfe      A       0.838854      0.052144       0.875847      0.041174      0.899639     0.061569
A_xgb_rfe      A       0.834720      0.053876       0.867023      0.047538      0.903291     0.052375
 A_rf_rfe      A       0.822601      0.059233       0.860963      0.049476      0.904562     0.058895
 A_xgb_l1      A       0.818881      0.063611       0.855080      0.055015      0.908915     0.052970
 B_xgb_l1      B       0.818881      0.063611       0.855080      0.055015      0.908915     0.052970
  A_rf_l1      A       0.817843      0.066246       0.858111      0.053719      0.902910     0.063849
  B_rf_l1      B  

In [ ]:
# Train the selected model and evaluate on Holdout 20%

# Use the selected best candidate
best_name = 'B_rf_l1'
print('Best model from CV:', best_name)

# Build the selected pipeline
final_model = make_method_pipeline(
    method='B',
    model_name='rf',
    selector_name='l1',
    scale_pos_weight=scale_pos_weight
)

# Train on the full training split
final_model.fit(X_raw, y_bin)

# Predict on holdout split
holdout_proba = final_model.predict_proba(X_holdout_raw)[:, 1]
holdout_pred_bin = (holdout_proba >= 0.5).astype(int)
holdout_pred = holdout_pred_bin + 1

# Compute metrics
holdout_auc = roc_auc_score(y_holdout_bin, holdout_proba)
holdout_f1 = f1_score(y_holdout, holdout_pred, average='macro')
holdout_acc = accuracy_score(y_holdout, holdout_pred)

print('\n=== Final Holdout (20%) Test ===')
print('Best model used:', best_name)
print(f'Holdout ROC-AUC: {holdout_auc:.4f}')
print(f'Holdout F1-macro: {holdout_f1:.4f}')
print(f'Holdout Accuracy: {holdout_acc:.4f}')
print('\nClassification Report (holdout):')
print(classification_report(y_holdout, holdout_pred, digits=4))

# Save results
holdout_result = pd.DataFrame({
    'id': df_holdout['id'].values,
    'gender_true': y_holdout.values,
    'gender_pred': holdout_pred,
    'proba_female': holdout_proba
})
holdout_result.to_csv('holdout_20_result.csv', index=False)
print('Saved: holdout_20_result.csv')

Best model from CV: B_rf_l1

=== Final Holdout (20%) Test ===
Best model used: B_rf_l1
Holdout ROC-AUC: 0.9455
Holdout F1-macro: 0.8739
Holdout Accuracy: 0.8941

Classification Report (holdout):
              precision    recall  f1-score   support

           1     0.9821    0.8730    0.9244        63
           2     0.7241    0.9545    0.8235        22

    accuracy                         0.8941        85
   macro avg     0.8531    0.9138    0.8739        85
weighted avg     0.9154    0.8941    0.8983        85

Saved: holdout_20_result.csv


In [ ]:
# L1 feature selection check: verify whether fb_friends / yt are retained
feature_step = final_model.named_steps['feature']
preprocess_step = final_model.named_steps['preprocess']
selector_step = final_model.named_steps['selector']

X_feat = feature_step.transform(X_raw)
_ = preprocess_step.transform(X_feat)
feature_names = np.array(preprocess_step.get_feature_names_out())
selected_mask = selector_step.get_support()
selected_features = feature_names[selected_mask]

print('=== L1 Selected Features ===')
print(f'Total selected: {selected_mask.sum()} / {len(feature_names)}')
for f in selected_features:
    print(' -', f)

fb_in = 'num__fb_friends' in selected_features
yt_in = 'num__yt' in selected_features
print('\nfb_friends selected:', fb_in)
print('yt selected:', yt_in)

=== L1 Selected Features ===
Total selected: 9 / 41
 - num__height
 - num__weight
 - num__fb_friends
 - num__bmi
 - num__intro_len
 - num__keyword_score
 - cat__star_element_earth
 - cat__height_bin_Unknown
 - cat__height_bin_mid

fb_friends selected: True
yt selected: False
